# Penn–Fudan Person Segmentation: Full Walkthrough

A step-by-step explanation of this project: task, data, architectures, experiment grid, training, metrics, result figures, error analysis, inference, and limitations. Result cells read the files in `artifacts/`; rerun them after the pipeline finishes to refresh.

**Run from the project root** using the `Python (Person Segmentation)` kernel. The project dependencies include `pandas` and `ipykernel`; install/update them with `python -m pip install -r requirements.txt` if needed. This notebook reads artifacts and visualizes them; it does not start, stop, or change training. Avoid costly evaluation while CPU training is active.

In [ ]:
from pathlib import Path
import json, csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown, Image as DisplayImage
ROOT=Path.cwd()
if not (ROOT/"config.yaml").exists():
    ROOT=next((p for p in [ROOT,*ROOT.parents] if (p/"config.yaml").exists()), ROOT)
ART=ROOT/"artifacts"
print("Project:",ROOT," | artifacts:",ART)

## 1. What the project does

Given an RGB street image, predict a binary mask: person pixel (1) or background pixel (0). The Penn–Fudan dataset has 170 images with instance masks (different IDs for different people); this project merges all nonzero IDs. It is **semantic segmentation**, not instance segmentation: it finds person pixels but does not label each person separately.

The fixed split is intended to be 102 train, 34 validation, and 34 test images. Train fits weights; validation selects checkpoints and tunes threshold; test is held out for final comparison. Images are resized to 256×256 (configured `squash`, which can change aspect ratio). Masks use nearest-neighbor resizing.

In [ ]:
split_path=ART/"splits.json"
split=json.loads(split_path.read_text()) if split_path.exists() else None
if split: display(pd.DataFrame([{"split":k,"images":len(v)} for k,v in split.items()]))
config_path=ROOT/"config.yaml"
print(config_path.read_text(encoding="utf-8") if config_path.exists() else "config.yaml missing")

### Look at actual images and masks

Instance IDs are shown in color; the binary target merges every nonzero ID into foreground.

In [ ]:
data=ROOT/"data"/"PennFudanPed"; imgs=data/"PNGImages"; masks=data/"PedMasks"
if not masks.exists(): masks=data/"PNGMasks"
if split and imgs.exists() and masks.exists():
    names=split["train"][:3]; fig,ax=plt.subplots(len(names),3,figsize=(10,3*len(names)),squeeze=False)
    for i,n in enumerate(names):
        im=Image.open(imgs/n).convert("RGB"); mp=masks/n
        if not mp.exists(): mp=masks/(Path(n).stem+"_mask.png")
        raw=np.asarray(Image.open(mp))
        ax[i,0].imshow(im); ax[i,0].set_title(n)
        ax[i,1].imshow(raw,cmap="tab20"); ax[i,1].set_title("Instance IDs")
        ax[i,2].imshow(raw>0,cmap="gray"); ax[i,2].set_title("Merged person mask")
        for a in ax[i]: a.axis("off")
    plt.tight_layout()
else: print("Local dataset not found; image examples skipped.")

## 2. Workflow diagram

Blue boxes represent training; orange boxes represent validation/test evaluation. The test set is not used to pick the best epoch.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
fig,ax=plt.subplots(figsize=(13,3.6)); ax.set_xlim(0,13); ax.set_ylim(0,3.7); ax.axis("off")
labels=["Images + masks","Frozen split\n102 / 34 / 34","Train-only\naugmentation","Model + loss\nAdamW / cosine LR","Best checkpoint\nvalidation IoU","Test metrics +\nrobustness"]
xs=[.2,2.3,4.4,6.5,8.6,10.7]
for i,(x,t) in enumerate(zip(xs,labels)):
    ax.add_patch(FancyBboxPatch((x,2),1.65,.9,boxstyle="round,pad=.04",fc="#dceafa" if i<5 else "#ffedcc",ec="#294968"))
    ax.text(x+.825,2.45,t,ha="center",va="center",fontsize=9)
    if i<5: ax.add_patch(FancyArrowPatch((x+1.67,2.45),(xs[i+1]-.05,2.45),arrowstyle="->",mutation_scale=13))
ax.text(3,0.8,"Validation: checkpoint selection and threshold tuning",ha="center")
ax.text(9.5,0.8,"Test: final scores, per-image errors, uncertainty, synthetic conditions",ha="center")
ax.set_title("Project pipeline"); plt.show()

## 3. Model architectures in `model.py`

**U-Net:** two Conv–BatchNorm–ReLU layers per block; four max-pooling encoder stages; a bottleneck; then four upsampling decoder stages. Encoder skip connections concatenate high-resolution detail into the decoder, helping recover object boundaries. The final 1×1 convolution produces one logit for each pixel.

**ResNet34-U-Net:** uses an ImageNet-pretrained ResNet34 encoder and a U-Net-style decoder. **DeepLabV3-ResNet50:** uses COCO/VOC pretrained segmentation weights and forms a binary person-vs-rest logit. `deeplab_ft` fine-tunes it; `deeplab_zeroshot` evaluates the pretrained weights without training.

The model file is compact because PyTorch layer blocks are reusable classes; the experiment registry chooses among model builders. This diagram is conceptual, not a layer-by-layer tensor dump.

In [ ]:
fig,ax=plt.subplots(figsize=(10,3.5)); ax.axis("off"); ax.set_xlim(0,10); ax.set_ylim(0,4)
for x,y,w,h,t,c in [(0.4,1.6,1.4,1,"Input\n256×256","#eee"),(2,1.3,1.4,1.6,"Encoder\n256→16","#b9d8f3"),(4,1.4,1.4,1.4,"Bottleneck","#b9d8f3"),(6,1.3,1.4,1.6,"Decoder\n16→256","#ffe0ad"),(8.2,1.6,1.4,1,"Person mask","#ffe0ad")]:
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle="round,pad=.04",fc=c,ec="#294968")); ax.text(x+w/2,y+h/2,t,ha="center",va="center")
for a,b in [(1.8,2),(3.4,4),(5.4,6),(7.4,8.2)]: ax.annotate("",(b,2),(a,2),arrowprops={"arrowstyle":"->"})
for y in [2.8,2.5,2.25]: ax.annotate("",(6,2.7),(2.7,y),arrowprops={"arrowstyle":"->","connectionstyle":"arc3,rad=-.3","color":"green"})
ax.text(5,3.5,"Skip connections carry spatial detail",ha="center",color="green"); ax.set_title("Simplified U-Net flow"); plt.show()

## 4. Augmentation, losses, training

Config: batch size 8 (eight 256×256 images per training step), 100 epochs, AdamW, learning rate 0.0003, weight decay 0.00001, cosine scheduler, threshold 0.5, automatic device selection. Augmentations include horizontal flip, affine scale/rotation/translation, color changes, blur, and noise; they apply to training only.

Some strong-augmentation experiments add synthetic distractor regions with probability 0.3. Targets remain unchanged so the network learns to ignore them. This evaluates synthetic robustness, not real-world glass/reflections.

**BCEWithLogits** is pixelwise classification loss. **Dice** rewards overlap. The combined loss is ((1-λ)BCE+λ(1-Dice)); `advanced` uses λ=0.5, while `advanced_ablation` uses λ=0 (BCE-only).

Each epoch evaluates validation IoU. `best.pt` stores the best validation-IoU weights; `progress.json` records live epoch progress; completed runs write `history.csv` and `summary.json`.

## 5. Experiment matrix

The configured seed is 20261103; the planned additional seeds are 1 and 2. Zero-shot DeepLab is evaluation-only. Three non-neural/simple baselines are also evaluated.

In [ ]:
experiments=[
("baseline","U-Net","none","BCE","reference"),
("augmented","U-Net","mild","BCE","augmentation"),
("advanced","U-Net","mild","BCE + 0.5 Dice","combined loss"),
("advanced_ablation","U-Net","mild","BCE; Dice weight 0","Dice ablation"),
("strong_aug","U-Net","strong","BCE","strong augmentation"),
("distract","U-Net","strong + distractors","BCE","distractor training"),
("resnet34","ResNet34-U-Net","strong","BCE","pretrained encoder"),
("resnet34_distract","ResNet34-U-Net","strong + distractors","BCE","encoder+distractors"),
("deeplab_ft","DeepLabV3-ResNet50","strong","BCE; LR 0.0001","fine tune"),
("deeplab_zeroshot","DeepLabV3-ResNet50","none","no training","zero-shot"),
("all_background","constant","none","none","simple baseline"),
("mean_training_mask","training-mask average","none","none","position prior"),
("pixel_random_forest","random forest","none","RF","classical baseline")]
display(pd.DataFrame(experiments,columns=["experiment","model","augmentation","loss","purpose"]))

## 6. Current run status and training curves

This reads progress and summary files live. A checkpoint by itself does not mean a run completed; evaluation requires `summary.json`.

In [ ]:
def readj(p):
    try: return json.loads(p.read_text(encoding="utf-8"))
    except Exception: return {}
runrows=[]
for p in sorted(ART.iterdir()) if ART.exists() else []:
    if p.is_dir():
        q=readj(p/"progress.json") or readj(p/"summary.json")
        if q: runrows.append({"run":p.name,"status":q.get("status","completed" if (p/"summary.json").exists() else "unknown"),"epoch":q.get("epoch"),"total":q.get("total_epochs",q.get("epochs")),"best epoch":q.get("best_epoch"),"best val IoU":q.get("best_validation_iou"),"device":readj(p/"summary.json").get("device"),"complete summary":(p/"summary.json").exists()})
run_status=pd.DataFrame(runrows); display(run_status)
print("Completed run summaries:",int(run_status["complete summary"].sum()) if len(run_status) else 0)
histories={p.parent.name:pd.read_csv(p) for p in ART.glob("*/history.csv")}
if histories:
    fig,ax=plt.subplots(1,2,figsize=(12,4))
    for n,d in histories.items():
        ax[0].plot(d.epoch,d.validation_loss,label=n); ax[1].plot(d.epoch,d.validation_iou,label=n)
    ax[0].set(title="Validation loss",xlabel="epoch"); ax[1].set(title="Validation IoU",xlabel="epoch",ylim=(0,1)); ax[1].legend(fontsize=7)
    plt.tight_layout()
else: print("History curves appear after a run completes.")

## 7. Metrics and what `metric_comparison.png` means

For person pixels: **precision=TP/(TP+FP)**, **recall=TP/(TP+FN)**, **F1=2PR/(P+R)**, **IoU=TP/(TP+FP+FN)**. The implementation's main metrics are pooled over all pixels. It also computes mean per-image IoU and a bootstrap confidence interval; these weight images equally and differ from pooled IoU.

`metric_comparison.png` is generated by `evaluate.py`: grouped bars for test precision, recall, F1, and pooled IoU by evaluated run, with a 0–1 y-axis. It is a valid quick comparison if generated from the current result CSV, but does not show uncertainty, runtime, or threshold-tuned IoU. Check timestamps; an old PNG can be stale. The code below reads the source table and image when present.

In [ ]:
resfile=ART/"results_summary.csv"
if resfile.exists():
    results=pd.read_csv(resfile); print("Results modified:",pd.Timestamp(resfile.stat().st_mtime,unit="s")); display(results)
else: results=None; print("results_summary.csv pending.")
metric_png=ART/"metric_comparison.png"
if metric_png.exists():
    print("Figure modified:",pd.Timestamp(metric_png.stat().st_mtime,unit="s")); display(DisplayImage(filename=str(metric_png)))
else: print("Figure appears after evaluation.")

### Aggregated results and paired comparisons

`results_table.csv` aggregates seeds as mean ± sample standard deviation and includes subgroup scores, runtime, parameters, and checkpoint size. Paired comparisons resample matched seeds and test images and report a confidence interval for second minus first. An interval crossing zero does not clearly separate the difference from zero; with three seeds/34 test images uncertainty may be substantial.

In [ ]:
for fn in ["results_table.csv","paired_comparison.csv"]:
    p=ART/fn
    print(fn)
    if p.exists(): display(pd.read_csv(p))
    else: print("Pending until final evaluation/table generation.")

## 8. Robustness and qualitative errors

Robustness conditions are synthetic reflection/shadow-like distractors. The distractor false-positive rate is predicted-person pixels inside the distractor region divided by distractor-region pixels. Lower is better for that specific error. Synthetic conditions do not prove real-world robustness.

Per-image CSVs reveal cases hidden by averages. Look at low-IoU images, false positives, and false negatives; inspect small/far people, occlusion, edges, and source-specific differences.

In [ ]:
rob=ART/"robustness_metrics.csv"
if rob.exists():
    robustness=pd.read_csv(rob); display(robustness.head(30))
    if {"run","condition","iou"}<=set(robustness):
        robustness.pivot_table(index="condition",columns="run",values="iou").plot(kind="bar",figsize=(12,4),ylim=(0,1),title="IoU by synthetic condition")
        plt.tight_layout(); plt.show()
else: print("robustness_metrics.csv pending.")
perdir=ART/"per_image"
files=sorted(perdir.glob("*.csv")) if perdir.exists() else []
if files:
    chosen=next((p for p in files if p.stem=="advanced"),files[0]); per=pd.read_csv(chosen)
    print("Per-image run:",chosen.stem); display(per.head(10))
    plt.hist(per.iou,bins=np.linspace(0,1,11)); plt.xlabel("IoU"); plt.ylabel("images"); plt.title("Per-image IoU"); plt.show()
else: print("Per-image evaluation files pending.")
for fn in ["eda.png","architecture.png","distractors.png","training_curves.png","qualitative.png","robustness.png"]:
    p=ART/"figures"/fn
    if p.exists(): print(fn,pd.Timestamp(p.stat().st_mtime,unit="s"))

## 9. Inference template

The model outputs logits. Apply sigmoid for person probabilities, then threshold to form a binary mask. Use the same resize and preprocessing as training. Uncomment and provide a local image path when a completed checkpoint exists.

In [ ]:
def predict_image(image_path, run="advanced", threshold=0.5):
    import sys, torch
    sys.path.insert(0,str(ROOT/"src"))
    from person_segmentation.model import load_trained
    from person_segmentation.utils import choose_device
    from person_segmentation.data import to_tensors
    model,_=load_trained(ART/run/"best.pt",choose_device("auto"))
    image=Image.open(image_path).convert("RGB").resize((256,256),Image.Resampling.BILINEAR)
    x,_=to_tensors(image,Image.new("L",image.size,0))
    with torch.no_grad(): prob=torch.sigmoid(model(x.unsqueeze(0).to(next(model.parameters()).device)))[0,0].cpu().numpy()
    fig,ax=plt.subplots(1,3,figsize=(10,3)); ax[0].imshow(image); ax[0].set_title("image")
    ax[1].imshow(prob,vmin=0,vmax=1); ax[1].set_title("probability")
    ax[2].imshow(prob>=threshold,cmap="gray"); ax[2].set_title("binary mask")
    for a in ax:a.axis("off")
    plt.show()
# predict_image(data/"PNGImages"/"FudanPed00001.png")

## 10. Files, reproducibility, and limitations

- `config.yaml`: shared settings; `splits.json`: frozen split.
- `src/person_segmentation/`: data, architectures, losses, training, evaluation, baselines, robustness.
- `artifacts/<run>/best.pt`: selected weights; `progress.json`: live progress; `history.csv`/ `summary.json`: completed training record.
- `results_summary.csv`: evaluated run metrics; `per_image/`: image scores; `predictions/`: predicted masks.
- `metric_comparison.png`: grouped test-metric chart; `figures/`: EDA, architecture, training, qualitative, and robustness figures.
- `report.md` and PDF: project write-up.

Reproducibility is limited by the small dataset and seed variation. The square resize can distort images. Pooled and mean-image IoU are different. Validation-tuned thresholds are separate from primary threshold 0.5 results. Synthetic distractors are not real reflection photographs. Results describe this fixed benchmark, not deployment readiness.

### Refresh final status
Rerun this last cell after training/evaluation to check which result files are available.

In [ ]:
for fn in ["results_summary.csv","results_table.csv","paired_comparison.csv","robustness_metrics.csv","metric_comparison.png"]:
    p=ART/fn
    print(fn, "AVAILABLE" if p.exists() else "PENDING", (pd.Timestamp(p.stat().st_mtime,unit="s") if p.exists() else ""))